# CATHOLIC UNIVERSITY OF RWANDA

# Deep Learning

## How computers learn useful patterns from examples

**Lecturer:** Fadl Musa  
**August 2026**

---

This notebook demonstrates the deep-learning workflow using the MNIST handwritten-digit dataset.

# How Deep Learning Works: Classifying Handwritten Digits

This beginner-friendly notebook demonstrates the complete deep-learning workflow:

1. Load and inspect data
2. Prepare images for a neural network
3. Define a model with hidden layers
4. Train by repeating prediction → loss → backpropagation → weight update
5. Evaluate on images the model did not train on
6. Inspect predictions and common mistakes

We use **MNIST**, a standard dataset of 70,000 grayscale images of handwritten digits (0–9). Each image is 28×28 pixels.

Dataset references:
- TensorFlow Datasets catalog: https://www.tensorflow.org/datasets/catalog/mnist
- Original MNIST page: http://yann.lecun.com/exdb/mnist/

The notebook is designed for Google Colab. It uses TensorFlow/Keras, which Colab commonly provides. If necessary, the next cell installs the required packages.

In [ ]:
# Install the main libraries if this Colab runtime does not already have them.
# The -q flag keeps installation output short.
!pip -q install tensorflow matplotlib

## 1. Import libraries and set reproducible seeds

NumPy handles arrays, Matplotlib displays images, and TensorFlow/Keras builds and trains the neural network. Setting seeds makes results more repeatable, although exact results can still vary between hardware environments.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

# Set seeds so that the random initialization and sampling are more repeatable.
np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)

## 2. Load the MNIST dataset

Keras downloads MNIST automatically the first time this cell runs. The training set is used to learn the weights. The test set is kept separate for an unbiased final check.

In [ ]:
# Load images and integer labels for training and testing.
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

print("Training images:", x_train.shape)
print("Training labels:", y_train.shape)
print("Test images:", x_test.shape)
print("Pixel range before scaling:", x_train.min(), "to", x_train.max())

## 3. Look at examples

Each image is a 28×28 matrix. The label is the digit shown in that image. The network will receive the pixel matrix, not the human-readable shape of the digit.

In [ ]:
# Display a small sample of images and their labels.
plt.figure(figsize=(8, 4))
for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.imshow(x_train[i], cmap="gray")
    plt.title(f"Label: {y_train[i]}")
    plt.axis("off")
plt.tight_layout()
plt.show()

## 4. Prepare the data

Pixel values originally range from 0 to 255. Dividing by 255 scales them to 0–1, which usually makes neural-network training more stable. We keep labels as integers because the model will use sparse categorical cross-entropy.

In [ ]:
# Normalize pixel intensities from [0, 255] to [0, 1].
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

print("Pixel range after scaling:", x_train.min(), "to", x_train.max())

## 5. Build a simple neural network

This model has:

- `Flatten`: turns each 28×28 image into a vector of 784 numbers.
- `Dense(128, relu)`: a hidden layer containing 128 neurons. ReLU is an activation function that adds useful nonlinearity.
- `Dense(10, softmax)`: ten output values, one for each digit. Softmax converts them into probabilities that sum to 1.

At initialization, the weights are mostly random. Learning will adjust them.

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28)),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax")
])

model.summary()

## 6. Configure the learning process

- The **optimizer** applies weight updates. Adam is a popular adaptive version of gradient descent.
- The **loss** measures how far the predicted probabilities are from the correct label.
- The **accuracy** metric reports the fraction of correct predictions.

Conceptually, training minimizes the loss by following gradients backward through the network (backpropagation).

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

## 7. Train the model

One **epoch** means the model has processed the training data once. During each batch, Keras performs a forward pass, calculates loss, computes gradients with backpropagation, and updates the weights.

In [ ]:
history = model.fit(
    x_train, y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

## 8. Visualize learning

Training and validation curves help us see whether performance improves. If training accuracy keeps rising while validation accuracy stops improving or falls, that can indicate overfitting.

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Training accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.title("Accuracy")

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("Loss")

plt.tight_layout()
plt.show()

## 9. Evaluate on the untouched test set

The test set represents new examples from the dataset. We use it only after training to estimate how well the learned function generalizes.

In [ ]:
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4%}")

## 10. Inspect predictions

The model outputs ten probabilities for each image. The predicted digit is the index with the largest probability.

In [ ]:
# Predict the first 12 test images.
probabilities = model.predict(x_test[:12], verbose=0)
predictions = np.argmax(probabilities, axis=1)

plt.figure(figsize=(10, 6))
for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.imshow(x_test[i], cmap="gray")
    color = "green" if predictions[i] == y_test[i] else "red"
    plt.title(f"True: {y_test[i]} | Pred: {predictions[i]}", color=color)
    plt.axis("off")
plt.tight_layout()
plt.show()

## 11. Find examples the model got wrong

Errors are informative. A digit may be ambiguous even to a person, or the model may have learned a shortcut that does not generalize.

In [ ]:
# Find the indexes where the prediction differs from the true label.
all_probabilities = model.predict(x_test, verbose=0)
all_predictions = np.argmax(all_probabilities, axis=1)
wrong = np.where(all_predictions != y_test)[0]

print("Number of incorrect test predictions:", len(wrong))

plt.figure(figsize=(10, 6))
for plot_index, image_index in enumerate(wrong[:12]):
    plt.subplot(3, 4, plot_index + 1)
    plt.imshow(x_test[image_index], cmap="gray")
    confidence = all_probabilities[image_index, all_predictions[image_index]]
    plt.title(f"True: {y_test[image_index]} | Pred: {all_predictions[image_index]}\nConfidence: {confidence:.2%}")
    plt.axis("off")
plt.tight_layout()
plt.show()

## 12. Summary: the deep-learning loop

- **Input:** pixel values representing a handwritten digit
- **Forward pass:** weighted layers transform pixels into class probabilities
- **Loss:** measures the prediction error
- **Backpropagation:** computes how each weight contributed to the error
- **Update:** the optimizer changes weights to reduce future loss
- **Generalization:** test accuracy checks performance on unseen examples

This is a small example, but the same core loop is used in much larger systems for images, text, audio, and other data.